In [ ]:
from random import randint, choice
from pprint import pprint


class Ship:
    def __init__(self, length, tp=1, x=None, y=None):
        self._x, self._y = x, y
        self._length = length
        self._tp = tp
        self._is_move = True
        self._cells = [1] * length
        self._coords, self._coords_around = [], []

    def set_start_coords(self, x, y):
        self._x, self._y = x, y

    def get_start_coords(self):
        return self._x, self._y

    def move(self, go):
        if self._is_move:
            if self._tp == 1:
                self._x += go
            else:
                self._y += go

    def is_collide(self, other):
        cor = []
        if self._tp == 1:
            for i in range(-1, 2):
                cor += [(x, self._y+i) for x in range(self._x-1, self._x+self._length+1)]
        else:
            for i in range(-1, 2):
                cor += [(self._x+i, y) for y in range(self._y-1, self._y+self._length+1)]

        if other._tp == 1:
            corr = [(x, other._y) for x in range(other._x, other._x+other._length)]
        else:
            corr = [(other._x, y) for y in range(other._y, other._y+other._length)]
        return len(set(cor) & set(corr)) > 0

    def is_out_pole(self, size=10):
        pole =[]
        for x in range(size):
            pole += [(x, y) for y in range(size)]

        if self._tp == 1:
            ship = [(x, self._y) for x in range(self._x, self._x + self._length)]
        else:
            ship = [(self._x, y) for y in range(self._y, self._y + self._length)]

        return bool(set(ship) - set(pole))

    def __getitem__(self, indx):
        return self._cells[indx]

    def __setitem__(self, indx, value):
        self._cells[indx] = value


class GamePole:
    def __init__(self, size=10):
        self._size = size
        self._ships = []

    def init(self):
        for r in (1, 1, 1, 1, 2, 2, 2, 3, 3, 4):
            k = 0
            s = Ship(r, tp=randint(1, 2))
            while True:
                if k == 40:
                    self._ships = []
                    self.init()
                    return
                x, y = randint(0, self._size - 1), randint(0, self._size - 1)
                s.set_start_coords(x, y)
                if not (s.is_out_pole(self._size) or any([s.is_collide(i) for i in self._ships])):
                    self._ships.append(s)
                    break
                k += 1

    def get_ships(self):
        return self._ships[:]

    def move_ships(self):
        for i, ship in enumerate(self._ships):
            if ship._is_move:
                step = choice([-1, 1])
                ship.move(step)
                if ship.is_out_pole(self._size) or any(
                        [ship.is_collide(s) for s in self._ships[:i] + self._ships[i + 1:]]):
                    ship.move(-(2 * step))
                    if ship.is_out_pole(self._size) or any(
                            [ship.is_collide(s) for s in self._ships[:i] + self._ships[i + 1:]]):
                        ship.move(step)

    def get_pole(self):
        pole = [[ 0 for _ in range(self._size)]for _ in range(self._size)]
        for s in self._ships:
            x, y, tp = s._x, s._y, s._tp
            for i in range(s._length):
                if tp == 1:
                    pole[y][x+i] = 1
                else:
                    pole[y+i][x] = 1
        return tuple([ tuple(i) for i in pole])

    def show(self):
        pprint(self.get_pole())
